# 09 — MULTI-CHECKPOINT_LOGIT-STACK

Preregistered experiment, cycle 09. Spec: `research_loop/preregistrations/09_MULTI-CHECKPOINT_LOGIT-STACK.yaml`.
Comparator: `07_TWITTER-ROBERTA_FINE-TUNE`. Structural base: `08` (multi-model loop, ensemble plumbing, artifact zip).

**What this run does.** Fine-tunes four heterogeneous checkpoints (355M / 435M / 125M / 125M) with 5-fold
stratified cross-validation *inside the train split only*, bags each backbone over its five folds at the logit
level, then compares seven preregistered combination candidates on validation. Validation is consumed by exactly
those seven candidate scores plus one threshold grid applied once to the winner.

**Hard rules enforced in code.**

1. The frozen split is read from the Kaggle foundation output only. The repo mirror `splits/split_assignments.csv`
   is a known-stale pre-repair file and is never read here.
2. A split-integrity assert runs before any training and raises on any count mismatch.
3. Cross-validation folds are drawn from the 2099 train rows only. Validation and test rows are never in a fold.
4. Test is unlocked only if the winner clears validation accuracy ≥ 0.8733 **and** validation ROC-AUC ≥ 0.9350.
   If the gate fails, no test metric is computed, printed, or saved. Probability artifacts are still exported.
5. One configuration, one threshold, one test evaluation, ever.
6. Exported probability CSVs carry exactly `row_id, split, y_true, y_prob` — no raw text, no text hash.

**Expected outcome.** INCONCLUSIVE. See the interpretation cell at the bottom before reading any number.

In [ ]:
# ============================================================
# 0. Kaggle notebook setup notes
# ============================================================

# This notebook is Kaggle-targeted by design (docs/REPLICATION_GUIDE.md).
#
# Attach as input the notebook output that contains research_foundation/:
#   /kaggle/input/notebooks/anthony73/extremism-research-dataset-creation-splits-nb/research_foundation/
#     - processed_dataset.csv
#     - split_assignments.csv
#     - dataset_manifest.json
#
# Accelerator: single T4 or P100. Internet: ON (four Hugging Face checkpoints are downloaded).
# If internet must stay off, attach the four checkpoints as Kaggle inputs and edit
# BACKBONES[*]["model_name"] to the local directories.

import os
from pathlib import Path

print("/kaggle/input exists:", Path("/kaggle/input").exists())
if Path("/kaggle/input").exists():
    print("Kaggle input datasets:")
    for input_path in sorted(Path("/kaggle/input").iterdir()):
        print(" -", input_path)

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import os
import gc
import json
import math
import time
import shutil
import hashlib
import platform
import datetime as dt
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 2. Central configuration and preregistered constants
# ============================================================

# ---------------------------------------------------------------- identity
TECHNIQUE_NAME = "MULTI-CHECKPOINT_LOGIT-STACK"
TECHNIQUE_ID = "09_MULTI-CHECKPOINT_LOGIT-STACK"
COMPARATOR_TECHNIQUE = "07_TWITTER-ROBERTA_FINE-TUNE"
PREREGISTRATION_PATH = "research_loop/preregistrations/09_MULTI-CHECKPOINT_LOGIT-STACK.yaml"

# ------------------------------------------------------- frozen data contract
# The full split identifier. Notebooks 01-08 declare only "split_v1"; that is a
# labelling defect in those notebooks, not a different split. Same frozen assignment.
SPLIT_VERSION = "split_v1_stratified_70_15_15_seed30"
DATASET_VERSION = "extremism_dataset_clean_v1"
RANDOM_SEED = 30

# (label 0, label 1) counts per split, taken from results_summary/foundation/.
# These are asserted before any training; a mismatch means the wrong split file was read.
EXPECTED_SPLIT_LABEL_COUNTS = {
    "train": (1309, 790),
    "validation": (281, 169),
    "test": (280, 170),
}
EXPECTED_SPLIT_ROW_COUNTS = {
    split_name: sum(counts) for split_name, counts in EXPECTED_SPLIT_LABEL_COUNTS.items()
}

# The Kaggle foundation directory emitted by 00_create_dataset_and_splits.ipynb.
KAGGLE_FOUNDATION_DIR = Path(
    "/kaggle/input/notebooks/anthony73/extremism-research-dataset-creation-splits-nb/research_foundation"
)
# Directory name of the stale committed mirror. Reading it is a hard error:
# it disagrees with the canonical assignment on 1420 rows and on 728 labels.
FORBIDDEN_SPLIT_MIRROR_DIRNAME = "splits"

# ------------------------------------------------------------- backbone set
LEARNING_RATES_LARGE = (1e-5, 2e-5)
LEARNING_RATES_BASE = (2e-5,)

BACKBONE_ORDER = ("B1", "B2", "B3", "B4")
BACKBONES = {
    "B1": {
        "model_name": "cardiffnlp/twitter-roberta-large-hate-latest",
        "size_class": "large",
        "params_millions": 355,
        "role": "domain and task matched; the scale arm of the hypothesis",
        "learning_rates": LEARNING_RATES_LARGE,
        "start_precision": "fp16",
        "droppable_on_time_budget": False,
    },
    "B2": {
        "model_name": "microsoft/deberta-v3-large",
        "size_class": "large",
        "params_millions": 435,
        "role": "different tokenizer and pretraining corpus; error decorrelation",
        "learning_rates": LEARNING_RATES_LARGE,
        # deberta-v3 has a documented fp16 AMP instability; fall back if loss goes non-finite.
        "start_precision": "fp16",
        "droppable_on_time_budget": True,
    },
    "B3": {
        "model_name": "cardiffnlp/twitter-roberta-base-hate-latest",
        "size_class": "base",
        "params_millions": 125,
        "role": "the champion checkpoint, present as both member and scale control",
        "learning_rates": LEARNING_RATES_BASE,
        "start_precision": "fp16",
        "droppable_on_time_budget": False,
    },
    "B4": {
        "model_name": "facebook/roberta-hate-speech-dynabench-r4-target",
        "size_class": "base",
        "params_millions": 125,
        "role": "same capacity as B3, adversarial hate-speech pretraining corpus",
        "learning_rates": LEARNING_RATES_BASE,
        "start_precision": "fp16",
        "droppable_on_time_budget": False,
    },
}

# ------------------------------------------- fixed training recipe (from 07)
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
MAX_LENGTH = 192
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.0          # 07 uses plain weighted CE; label smoothing is 08's.
DATALOADER_NUM_WORKERS = 2
DROP_LAST_TRAIN_BATCH = False

# --------------------------------------- train-side cross-validation protocol
N_FOLDS = 5
MAX_EPOCHS = 4
EPOCH_PATIENCE = 1             # stop when epochs-without-improvement exceeds this
FOLD_SELECTION_METRIC = "accuracy"

# ----------------------------------------------------- decision and selection
POSITIVE_LABEL = 1
POSITIVE_CLASS_INDEX = 1
NEGATIVE_CLASS_INDEX = 0
# Candidates are scored against each other at this fixed threshold; only the
# winner gets the threshold grid. That keeps the validation budget at
# 7 candidate scores + 1 sweep, as preregistered.
DEFAULT_DECISION_THRESHOLD = 0.50
THRESHOLD_GRID = [round(float(x), 3) for x in np.linspace(0.05, 0.95, 181)]
THRESHOLD_OBJECTIVE_METRIC = "accuracy"   # the promotion metric is test accuracy

CANDIDATE_IDS = ("C1", "C2", "C3", "C4", "C5", "C6", "C7")
CANDIDATE_TO_SINGLE_BACKBONE = {"C1": "B1", "C2": "B2", "C3": "B3", "C4": "B4"}

# ------------------------------------------------------------ validation gate
# Both must hold to unlock test. 0.8733 is the validation accuracy that the
# repository's measured val->test offset (+2.83pp) maps to a predicted 0.9016.
VALIDATION_GATE_MIN_ACCURACY = 0.8733
VALIDATION_GATE_MIN_ROC_AUC = 0.9350

# --------------------------------------------------------- robustness budgets
PRECISION_FALLBACK_ORDER = ("fp16", "bf16", "fp32")
MIN_OOF_ACCURACY_FOR_ENSEMBLE = 0.78      # below this, exclude from C5/C6/C7
MIN_MEMBERS_FOR_ENSEMBLE = 2
SECONDS_PER_HOUR = 3600
MAX_ELAPSED_SECONDS_BEFORE_DROPPING_OPTIONAL_BACKBONE = 4 * SECONDS_PER_HOUR

# ------------------------------------------------- stack and TF-IDF ablation
STACK_INVERSE_REGULARIZATION = 1.0
STACK_MAX_ITERATIONS = 1000
TFIDF_MAX_FEATURES = 50000
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 2
TFIDF_INVERSE_REGULARIZATION = 1.0

# ------------------------------------------------- comparator reference values
# Committed numbers for 07, used only for reporting context. Never for selection.
COMPARATOR_TEST_ACCURACY = 0.8889
COMPARATOR_TEST_BALANCED_ACCURACY = 0.8853
COMPARATOR_VALIDATION_ACCURACY = 0.8578
COMPARATOR_VALIDATION_ROC_AUC = 0.9257
PROJECT_TARGET_TEST_ACCURACY = 0.9000
HOLM_DETECTABLE_TEST_ACCURACY = 0.9333    # +4.44pp: what significance would need at m=9
BALANCED_ACCURACY_NON_INFERIORITY_MARGIN = 0.005
FPR_FLAG_THRESHOLD = 0.10 + 0.02

# ------------------------------------------------------------ optional extras
# 3-seed replicate of the best single backbone (prereg: "optional if time permits").
# Off by default: it costs one extra full CV run per additional seed.
RUN_OPTIONAL_SEED_REPLICATE = False
OPTIONAL_REPLICATE_SEEDS = (31, 32)

CONFIG = {
    # Experiment identity
    "project_name": "extremism_text_classification",
    "technique_name": TECHNIQUE_NAME,
    "technique_id": TECHNIQUE_ID,
    "model_family": "multi_checkpoint_logit_stack_fine_tuned_transformers",
    "feature_family": "contextual_transformer_sequence_representation",
    "dataset_version": DATASET_VERSION,
    "split_version": SPLIT_VERSION,
    "random_seed": RANDOM_SEED,
    "preregistration": PREREGISTRATION_PATH,
    "comparator": COMPARATOR_TECHNIQUE,

    # Paths. Leave None for auto-discovery under the Kaggle foundation directory.
    "processed_dataset_path": None,
    "split_assignments_path": None,
    "dataset_manifest_path": None,

    # Required columns from 00_create_dataset_and_splits.ipynb
    "id_col": "row_id",
    "text_col": "text",
    "label_col": "label",
    "split_col": "split",
    "positive_label": POSITIVE_LABEL,

    # Output
    "output_root": "/kaggle/working/experiments",
    "overwrite_output_dir": True,

    # Backbones and recipe
    "backbones": {key: spec["model_name"] for key, spec in BACKBONES.items()},
    "fixed_recipe": {
        "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_length": MAX_LENGTH,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "loss": "weighted_cross_entropy(class_weight=balanced)",
        "label_smoothing": LABEL_SMOOTHING,
        "seed": RANDOM_SEED,
    },
    "train_side_cross_validation": {
        "n_folds": N_FOLDS,
        "stratified": True,
        "scope": "TRAIN SPLIT ONLY (2099 rows); validation and test are never in a fold",
        "epoch_selection": "held-out fold, metric=accuracy, max_epochs=4, patience=1",
        "lr_selection": "held-out fold OOF accuracy; large={1e-5,2e-5}, base={2e-5}",
        "inference": "mean of the 5 fold models' logits (fold-bagging), per backbone",
    },

    # Probability / thresholding
    "threshold_selection": {
        "metric": THRESHOLD_OBJECTIVE_METRIC,
        "threshold_grid": THRESHOLD_GRID,
        "applied_to": "the winning candidate only, once",
    },

    # Validation gate
    "validation_gate": {
        "min_accuracy": VALIDATION_GATE_MIN_ACCURACY,
        "min_roc_auc": VALIDATION_GATE_MIN_ROC_AUC,
        "if_not_met": "no test unlock; validation-only result set and probability artifacts only",
    },

    # Error analysis. Text previews are never written: sanitized artifacts only.
    "error_analysis": {
        "examples_per_bucket": 30,
        "near_threshold_margin": 0.05,
        "include_text_preview": False,
        "text_preview_chars": 0,
    },
}

RUN_ID = f"{TECHNIQUE_ID}_{dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
RUN_START_TIME = time.time()
OUTPUT_DIR = Path(CONFIG["output_root"]) / TECHNIQUE_ID
PROBS_DIR = OUTPUT_DIR / "probs"

if CONFIG["overwrite_output_dir"] and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for subdir in [
    OUTPUT_DIR,
    OUTPUT_DIR / "ablation",
    OUTPUT_DIR / "training_logs",
    OUTPUT_DIR / "plots",
    OUTPUT_DIR / "error_analysis",
    PROBS_DIR,
]:
    subdir.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Output directory:", OUTPUT_DIR)
print("Probability export directory:", PROBS_DIR)
print("Split version:", SPLIT_VERSION)
print("Threshold objective:", THRESHOLD_OBJECTIVE_METRIC)
print("Validation gate: accuracy >=", VALIDATION_GATE_MIN_ACCURACY, "AND roc_auc >=", VALIDATION_GATE_MIN_ROC_AUC)

In [ ]:
# ============================================================
# 3. Reproducibility and helper functions
# ============================================================

def set_global_seed(seed: int) -> None:
    """Seed python, numpy and torch. cudnn.benchmark is left on: determinism here costs more than it buys."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def make_json_safe(obj):
    """Convert numpy/torch/path objects into JSON-serializable Python objects."""
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if math.isnan(float(obj)):
            return None
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    try:
        if pd.isna(obj) and not isinstance(obj, (list, tuple, dict, np.ndarray)):
            return None
    except Exception:
        pass
    return obj


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(make_json_safe(obj), f, indent=2)


def append_jsonl(record: dict, path: Path) -> None:
    """Append one JSON record per line. Used for the validation candidate log."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(make_json_safe(record)) + "\n")


def sha256_file(path: Path, n_chars: int = 16) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()[:n_chars]


def find_foundation_file(file_name: str) -> Path:
    """
    Locate a foundation file emitted by notebook 00.

    Searches the pinned Kaggle foundation directory first, then the rest of
    /kaggle/input and /kaggle/working. The repository mirror splits/split_assignments.csv
    is refused outright: it is the stale pre-repair file that disagrees with the
    canonical assignment on 1420 rows and 728 labels, and using it would silently
    invalidate every number this notebook produces.
    """
    search_roots = [KAGGLE_FOUNDATION_DIR, Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in search_roots:
        if root.exists():
            if (root / file_name).exists():
                matches.append(root / file_name)
            matches.extend(root.rglob(file_name))

    matches = [p for p in sorted(set(matches), key=lambda p: (len(str(p)), str(p)))]
    matches = [p for p in matches if p.parent.name != FORBIDDEN_SPLIT_MIRROR_DIRNAME]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {file_name} under {[str(r) for r in search_roots]}. "
            "Attach the research_foundation output of 00_create_dataset_and_splits.ipynb, "
            "or set the explicit path in CONFIG."
        )
    if len(matches) > 1:
        print(f"Multiple matches for {file_name}; using the shortest path:")
        for match in matches[:10]:
            print(" -", match)
    return matches[0]


def free_gpu_memory() -> None:
    """Drop cached CUDA blocks between models. A 435M backbone plus fold models must not accumulate."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


set_global_seed(RANDOM_SEED)
save_json(CONFIG, OUTPUT_DIR / "config.json")
print("Seed set to", RANDOM_SEED)

In [ ]:
# ============================================================
# 4. Load processed dataset, split assignments, and manifest
# ============================================================

processed_dataset_path = (
    Path(CONFIG["processed_dataset_path"])
    if CONFIG["processed_dataset_path"] is not None
    else find_foundation_file("processed_dataset.csv")
)
split_assignments_path = (
    Path(CONFIG["split_assignments_path"])
    if CONFIG["split_assignments_path"] is not None
    else find_foundation_file("split_assignments.csv")
)
dataset_manifest_path = (
    Path(CONFIG["dataset_manifest_path"])
    if CONFIG["dataset_manifest_path"] is not None
    else find_foundation_file("dataset_manifest.json")
)

if split_assignments_path.parent.name == FORBIDDEN_SPLIT_MIRROR_DIRNAME:
    raise RuntimeError(
        f"Refusing to read the stale committed split mirror at {split_assignments_path}. "
        "Use the Kaggle foundation output from 00_create_dataset_and_splits.ipynb."
    )

processed_df = pd.read_csv(processed_dataset_path)
splits_df = pd.read_csv(split_assignments_path)
with open(dataset_manifest_path, "r") as f:
    dataset_manifest = json.load(f)

# Deliberately no display of rows: saved outputs must never carry dataset text or row ids.
print("Processed dataset path:", processed_dataset_path)
print("Split assignments path:", split_assignments_path)
print("Dataset manifest path:", dataset_manifest_path)
print("Processed dataset shape:", processed_df.shape)
print("Split assignments shape:", splits_df.shape)
print("Processed dataset columns:", sorted(processed_df.columns.tolist()))
print("Split assignment columns:", sorted(splits_df.columns.tolist()))

In [ ]:
# ============================================================
# 5. Validate schema and apply the canonical frozen split
# ============================================================

id_col = CONFIG["id_col"]
text_col = CONFIG["text_col"]
label_col = CONFIG["label_col"]
split_col = CONFIG["split_col"]
positive_label = CONFIG["positive_label"]

missing_dataset_cols = {id_col, text_col, label_col} - set(processed_df.columns)
if missing_dataset_cols:
    raise ValueError(f"processed_dataset.csv is missing required columns: {missing_dataset_cols}")

missing_split_cols = {id_col, split_col} - set(splits_df.columns)
if missing_split_cols:
    raise ValueError(f"split_assignments.csv is missing required columns: {missing_split_cols}")

merged = processed_df.merge(
    splits_df[[id_col, split_col]],
    on=id_col,
    how="left",
    validate="one_to_one",
)

if merged[split_col].isna().any():
    n_missing = int(merged[split_col].isna().sum())
    raise ValueError(f"{n_missing} dataset rows have no split assignment.")

# Text is preserved exactly as produced by preprocessing: casing, punctuation,
# hashtags and obfuscation all carry signal for a transformer.
merged[text_col] = merged[text_col].fillna("").astype(str)
merged[label_col] = merged[label_col].astype(int)
merged[split_col] = merged[split_col].astype(str).str.lower().str.strip()

allowed_splits = {"train", "validation", "test"}
unexpected_splits = sorted(set(merged[split_col]) - allowed_splits)
if unexpected_splits:
    raise ValueError(f"Unexpected split labels found: {unexpected_splits}")

train_df = merged[merged[split_col] == "train"].reset_index(drop=True)
val_df = merged[merged[split_col] == "validation"].reset_index(drop=True)
test_df = merged[merged[split_col] == "test"].reset_index(drop=True)
SPLIT_FRAMES = {"train": train_df, "validation": val_df, "test": test_df}

y_train = train_df[label_col].values.astype(int)
y_val = val_df[label_col].values.astype(int)
y_test = test_df[label_col].values.astype(int)

split_audit = (
    merged.groupby([split_col, label_col]).size().reset_index(name="count")
)
split_audit["rate_within_split"] = split_audit.groupby(split_col)["count"].transform(lambda s: s / s.sum())
split_audit.to_csv(OUTPUT_DIR / "split_label_distribution_used.csv", index=False)

print("Split sizes:", {name: len(frame) for name, frame in SPLIT_FRAMES.items()})
print(split_audit.to_string(index=False))

In [ ]:
# ============================================================
# 6. Split integrity assert  --  hard gate before any training
# ============================================================

def assert_split_integrity(split_frames: Dict[str, pd.DataFrame]) -> Dict[str, dict]:
    """
    Verify the loaded split is the frozen one before a single GPU second is spent.

    Checks, per split: exact row count, exact (label 0, label 1) counts, no duplicate
    row_ids. Across splits: pairwise disjoint row_id sets. Every observed mismatch is
    collected so the error message shows all of them at once, then RuntimeError is raised.
    """
    problems: List[str] = []
    observed: Dict[str, dict] = {}

    for split_name in ("train", "validation", "test"):
        if split_name not in split_frames:
            problems.append(f"split '{split_name}' is entirely missing from the merged data")
            continue
        frame = split_frames[split_name]
        n_rows = len(frame)
        n_negative = int((frame[label_col] == 0).sum())
        n_positive = int((frame[label_col] == 1).sum())
        observed[split_name] = {
            "n_rows": n_rows,
            "label_counts": [n_negative, n_positive],
        }

        expected_rows = EXPECTED_SPLIT_ROW_COUNTS[split_name]
        if n_rows != expected_rows:
            problems.append(f"{split_name}: row count {n_rows} != expected {expected_rows}")

        expected_labels = EXPECTED_SPLIT_LABEL_COUNTS[split_name]
        if (n_negative, n_positive) != expected_labels:
            problems.append(
                f"{split_name}: (label 0, label 1) counts ({n_negative}, {n_positive}) "
                f"!= expected {expected_labels}"
            )

        n_duplicate_ids = int(frame[id_col].duplicated().sum())
        if n_duplicate_ids:
            problems.append(f"{split_name}: {n_duplicate_ids} duplicate row_id values")

    id_sets = {
        name: set(frame[id_col].tolist())
        for name, frame in split_frames.items()
        if name in ("train", "validation", "test")
    }
    for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
        if left in id_sets and right in id_sets:
            overlap = id_sets[left] & id_sets[right]
            if overlap:
                problems.append(f"{left} and {right} share {len(overlap)} row_ids; splits must be disjoint")

    if problems:
        raise RuntimeError(
            "SPLIT INTEGRITY ASSERT FAILED -- refusing to train.\n"
            + "\n".join(f"  - {p}" for p in problems)
            + "\n\nExpected counts come from results_summary/foundation/ for "
            + SPLIT_VERSION
            + ".\nThe most likely cause is reading the stale repository mirror "
            + "splits/split_assignments.csv (train 1121/978, validation 241/209, test 240/210) "
            + "instead of the Kaggle foundation output of 00_create_dataset_and_splits.ipynb. "
            + "Fix the input, do not adjust the expected counts."
        )
    return observed


observed_split_shape = assert_split_integrity(SPLIT_FRAMES)
save_json(
    {
        "split_version": SPLIT_VERSION,
        "dataset_version": DATASET_VERSION,
        "expected_label_counts": EXPECTED_SPLIT_LABEL_COUNTS,
        "observed": observed_split_shape,
        "split_assignments_path": str(split_assignments_path),
        "split_assignments_sha256_16": sha256_file(split_assignments_path),
        "processed_dataset_path": str(processed_dataset_path),
        "processed_dataset_sha256_16": sha256_file(processed_dataset_path),
        "status": "PASS",
    },
    OUTPUT_DIR / "split_integrity_check.json",
)

print("SPLIT INTEGRITY ASSERT: PASS")
for split_name, shape in observed_split_shape.items():
    print(f"  {split_name}: {shape['n_rows']} rows, (label0, label1) = {tuple(shape['label_counts'])}")
print("train / validation / test row_id sets are pairwise disjoint.")

In [ ]:
# ============================================================
# 7. Metric, threshold, and prediction utilities
# ============================================================

# compute_binary_metrics is copied unchanged from 07 (cell 6) and is the canonical
# implementation shared with tools/metrics_core.py. Do not substitute 08's
# evaluate_binary_predictions: it drops support/f1_macro/f1_weighted and renames
# false_positive_rate/false_negative_rate to fpr/fnr, which the results schema rejects.

def safe_roc_auc(y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob)
    except Exception:
        return np.nan


def safe_pr_auc(y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return average_precision_score(y_true, y_prob)
    except Exception:
        return np.nan


def compute_binary_metrics(y_true, y_prob, threshold, positive_label=1):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    labels = [0, 1]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=labels).ravel()

    metrics = {
        "threshold": float(threshold),
        "support": int(len(y_true)),
        "positive_support": int((y_true == positive_label).sum()),
        "negative_support": int((y_true != positive_label).sum()),
        "positive_rate": float((y_true == positive_label).mean()),

        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),

        "positive_precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "positive_recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "positive_f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),

        "roc_auc": float(safe_roc_auc(y_true, y_prob)),
        "pr_auc": float(safe_pr_auc(y_true, y_prob)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan,
    }
    return metrics


def select_threshold(y_true, y_prob, thresholds, metric=THRESHOLD_OBJECTIVE_METRIC):
    """Sweep the threshold grid once and return (best_threshold, full_sweep_frame)."""
    rows = []
    for threshold in thresholds:
        rows.append(compute_binary_metrics(y_true, y_prob, threshold, CONFIG["positive_label"]))
    threshold_df = pd.DataFrame(rows)
    threshold_df = threshold_df.sort_values(
        by=[metric, "positive_recall", "positive_precision", "balanced_accuracy"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    return float(threshold_df.iloc[0]["threshold"]), threshold_df


def margins_to_probabilities(margins) -> np.ndarray:
    """
    Map decision margins (logit_positive - logit_negative) to positive-class probabilities.

    For a two-logit softmax head, softmax(logits)[1] == sigmoid(logit_1 - logit_0), so the
    scalar margin is a lossless summary of the head and is the natural quantity to average
    at the "logit level" and to feed the stack. Computed via the numerically stable
    expit-equivalent form rather than exp() to avoid overflow on large margins.
    """
    margins = np.asarray(margins, dtype=float)
    return np.where(
        margins >= 0,
        1.0 / (1.0 + np.exp(-np.clip(margins, -60.0, 60.0))),
        np.exp(np.clip(margins, -60.0, 60.0)) / (1.0 + np.exp(np.clip(margins, -60.0, 60.0))),
    )


def make_sanitized_prediction_frame(split_frame: pd.DataFrame, y_prob, threshold: float) -> pd.DataFrame:
    """Build a prediction frame with no raw text and no text hash: row_id, label, probability, outcome."""
    out = pd.DataFrame(
        {
            "row_id": split_frame[id_col].values,
            "split": split_frame[split_col].values,
            "y_true": split_frame[label_col].values.astype(int),
            "y_prob": np.asarray(y_prob, dtype=float),
        }
    )
    out["threshold"] = float(threshold)
    out["y_pred"] = (out["y_prob"] >= float(threshold)).astype(int)
    out["correct"] = out["y_true"] == out["y_pred"]
    out["error_type"] = "correct"
    out.loc[(out["y_true"] == 0) & (out["y_pred"] == 1), "error_type"] = "false_positive"
    out.loc[(out["y_true"] == 1) & (out["y_pred"] == 0), "error_type"] = "false_negative"
    out["abs_distance_to_threshold"] = (out["y_prob"] - float(threshold)).abs()
    near_margin = CONFIG["error_analysis"]["near_threshold_margin"]
    out["confidence_bucket"] = np.where(
        out["abs_distance_to_threshold"] <= near_margin,
        "near_threshold",
        np.where((out["y_prob"] >= 0.90) | (out["y_prob"] <= 0.10), "high_confidence", "moderate_confidence"),
    )
    return out


print("Metric utilities ready. Threshold objective:", THRESHOLD_OBJECTIVE_METRIC)
print("Threshold grid size:", len(THRESHOLD_GRID), "from", THRESHOLD_GRID[0], "to", THRESHOLD_GRID[-1])

In [ ]:
# ============================================================
# 8. Transformer dataset, loss, and AMP utilities
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

AMP_DTYPE_BY_PRECISION = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": None}


class TextClassificationDataset(Dataset):
    """Holds raw strings; tokenization happens per batch so padding is dynamic (faster than padding to the split max)."""

    def __init__(self, texts, labels=None):
        self.texts = [str(x) for x in texts]
        self.labels = None if labels is None else np.asarray(labels).astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx]}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


def make_collate_fn(tokenizer, max_length: int):
    def collate_fn(batch):
        encoded = tokenizer(
            [b["text"] for b in batch],
            truncation=True,
            padding=True,
            max_length=int(max_length),
            return_tensors="pt",
        )
        if "label" in batch[0]:
            encoded["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        return encoded
    return collate_fn


def make_dataloader(df_part: pd.DataFrame, tokenizer, batch_size: int, shuffle: bool = False) -> DataLoader:
    """Build a DataLoader over a split/fold frame using the fixed MAX_LENGTH recipe."""
    dataset = TextClassificationDataset(
        texts=df_part[text_col].tolist(),
        labels=df_part[label_col].tolist(),
    )
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=shuffle,
        collate_fn=make_collate_fn(tokenizer, MAX_LENGTH),
        num_workers=DATALOADER_NUM_WORKERS,
        drop_last=DROP_LAST_TRAIN_BATCH if shuffle else False,
        pin_memory=torch.cuda.is_available(),
    )


def compute_balanced_class_weights(y_values) -> torch.Tensor:
    """Balanced class weights for nn.CrossEntropyLoss, computed on the rows actually being fit."""
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=np.asarray(y_values).astype(int),
    )
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def amp_autocast(precision: str):
    """Autocast context for the requested precision; a no-op context when precision is fp32 or on CPU."""
    dtype = AMP_DTYPE_BY_PRECISION[precision]
    enabled = dtype is not None and DEVICE.type == "cuda"
    if not enabled:
        return torch.autocast(device_type="cpu", enabled=False)
    return torch.autocast(device_type="cuda", dtype=dtype)


def make_grad_scaler(precision: str):
    """GradScaler is only meaningful for fp16; bf16 and fp32 run unscaled."""
    enabled = precision == "fp16" and DEVICE.type == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def precision_fallback_sequence(start_precision: str) -> List[str]:
    """
    Precisions to try, in order, starting at the backbone's preferred one.

    bf16 is dropped when the GPU cannot do it (T4 is Turing: fp16 yes, bf16 no),
    so on a T4 the deberta fallback chain collapses to fp16 -> fp32.
    """
    start_index = PRECISION_FALLBACK_ORDER.index(start_precision)
    sequence = list(PRECISION_FALLBACK_ORDER[start_index:])
    if "bf16" in sequence:
        bf16_supported = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
        if not bf16_supported:
            sequence.remove("bf16")
    return sequence


print("Precision fallback chain for a large backbone:", precision_fallback_sequence("fp16"))

In [ ]:
# ============================================================
# 9. Fold-level training and inference
# ============================================================

class NanLossError(RuntimeError):
    """Raised when a training step produces a non-finite loss (the known deberta-v3 fp16 AMP failure)."""


@torch.no_grad()
def predict_margins(model, loader) -> np.ndarray:
    """Return the scalar decision margin (logit_positive - logit_negative) for every row in loader, in order."""
    model.eval()
    margins = []
    for batch in loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        logits = model(**inputs).logits.float()
        margins.append((logits[:, POSITIVE_CLASS_INDEX] - logits[:, NEGATIVE_CLASS_INDEX]).cpu().numpy())
    return np.concatenate(margins).astype(float)


def train_fold_model(
    model_name: str,
    learning_rate: float,
    precision: str,
    seed: int,
    fold_train_df: pd.DataFrame,
    fold_holdout_df: pd.DataFrame,
    run_label: str,
) -> dict:
    """
    Fine-tune one fold model and return the best epoch's margins for holdout / validation / test.

    Epoch selection uses the held-out TRAIN fold only (accuracy at DEFAULT_DECISION_THRESHOLD,
    at most MAX_EPOCHS epochs, patience EPOCH_PATIENCE). Validation and test rows are only ever
    passed through the model for inference, never for selection at this level.

    No model weights are retained. Instead the margins produced at each improving epoch are
    captured immediately, which removes the need to hold a 435M-parameter state-dict copy in
    host RAM while the next fold trains. The model is deleted and CUDA cache cleared before return.

    Raises NanLossError so the caller can fall back fp16 -> bf16 -> fp32.
    """
    set_global_seed(seed)
    fold_start = time.time()

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "non_extremist", 1: "extremist"},
        label2id={"non_extremist": 0, "extremist": 1},
        ignore_mismatched_sizes=True,
    )
    model.to(DEVICE)

    train_loader = make_dataloader(fold_train_df, tokenizer, TRAIN_BATCH_SIZE, shuffle=True)
    holdout_loader = make_dataloader(fold_holdout_df, tokenizer, EVAL_BATCH_SIZE)
    validation_loader = make_dataloader(val_df, tokenizer, EVAL_BATCH_SIZE)
    test_loader = make_dataloader(test_df, tokenizer, EVAL_BATCH_SIZE)

    criterion = nn.CrossEntropyLoss(
        weight=compute_balanced_class_weights(fold_train_df[label_col].values),
        label_smoothing=LABEL_SMOOTHING,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(learning_rate), weight_decay=WEIGHT_DECAY)

    update_steps_per_epoch = max(1, math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
    total_update_steps = update_steps_per_epoch * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_update_steps),
        num_training_steps=total_update_steps,
    )
    scaler = make_grad_scaler(precision)

    y_holdout = fold_holdout_df[label_col].values.astype(int)
    best_epoch = None
    best_holdout_accuracy = -np.inf
    best_margins = {"holdout": None, "validation": None, "test": None}
    epochs_without_improvement = 0
    epoch_logs = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        for step, batch in enumerate(train_loader, start=1):
            labels = batch["labels"].to(DEVICE)
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}

            with amp_autocast(precision):
                outputs = model(**inputs)
                loss = criterion(outputs.logits.float(), labels)

            if not torch.isfinite(loss):
                del model, optimizer, scheduler, scaler
                free_gpu_memory()
                raise NanLossError(
                    f"{run_label}: non-finite training loss at epoch {epoch}, step {step} under {precision}"
                )

            scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            running_loss += float(loss.detach().cpu())

            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        mean_loss = running_loss / max(1, len(train_loader))
        if not math.isfinite(mean_loss):
            del model, optimizer, scheduler, scaler
            free_gpu_memory()
            raise NanLossError(f"{run_label}: non-finite mean epoch loss at epoch {epoch} under {precision}")

        holdout_margin = predict_margins(model, holdout_loader)
        holdout_pred = (margins_to_probabilities(holdout_margin) >= DEFAULT_DECISION_THRESHOLD).astype(int)
        holdout_accuracy = float(accuracy_score(y_holdout, holdout_pred))

        epoch_logs.append(
            {
                "run_label": run_label,
                "epoch": epoch,
                "train_loss_mean": mean_loss,
                "holdout_accuracy": holdout_accuracy,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
                "precision": precision,
            }
        )
        print(
            f"{run_label} | epoch {epoch}/{MAX_EPOCHS} | loss={mean_loss:.4f} | "
            f"holdout_acc={holdout_accuracy:.4f} | precision={precision}"
        )

        if holdout_accuracy > best_holdout_accuracy:
            best_holdout_accuracy = holdout_accuracy
            best_epoch = epoch
            best_margins["holdout"] = holdout_margin
            best_margins["validation"] = predict_margins(model, validation_loader)
            best_margins["test"] = predict_margins(model, test_loader)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement > EPOCH_PATIENCE:
                print(f"{run_label} | early stop after epoch {epoch}; best epoch was {best_epoch}")
                break

    del model, optimizer, scheduler, scaler
    del train_loader, holdout_loader, validation_loader, test_loader, tokenizer
    free_gpu_memory()

    return {
        "run_label": run_label,
        "best_epoch": best_epoch,
        "best_holdout_accuracy": float(best_holdout_accuracy),
        "holdout_margin": best_margins["holdout"],
        "validation_margin": best_margins["validation"],
        "test_margin": best_margins["test"],
        "epoch_logs": epoch_logs,
        "runtime_seconds": time.time() - fold_start,
    }

In [ ]:
# ============================================================
# 10. Backbone cross-validation driver with robustness fallbacks
# ============================================================

def run_backbone_at_learning_rate(backbone_key: str, learning_rate: float, precision: str, seed: int) -> dict:
    """
    Run the full N_FOLDS stratified CV for one (backbone, learning rate) configuration.

    Folds are drawn from the train split only. Each fold contributes: its held-out rows'
    margins (assembled into the complete OOF vector over the 2099 train rows) and its
    margins on validation and test, which are averaged across folds -- fold-bagging at the
    logit level, as preregistered.
    """
    spec = BACKBONES[backbone_key]
    oof_margin = np.full(len(train_df), np.nan, dtype=float)
    validation_margins: List[np.ndarray] = []
    test_margins: List[np.ndarray] = []
    fold_records: List[dict] = []
    epoch_logs: List[dict] = []

    folder = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    config_start = time.time()

    for fold_index, (fit_index, holdout_index) in enumerate(
        folder.split(np.zeros(len(train_df)), y_train), start=1
    ):
        run_label = f"{backbone_key}_lr{learning_rate:g}_{precision}_fold{fold_index}"
        fold_result = train_fold_model(
            model_name=spec["model_name"],
            learning_rate=learning_rate,
            precision=precision,
            seed=seed,
            fold_train_df=train_df.iloc[fit_index].reset_index(drop=True),
            fold_holdout_df=train_df.iloc[holdout_index].reset_index(drop=True),
            run_label=run_label,
        )
        oof_margin[holdout_index] = fold_result["holdout_margin"]
        validation_margins.append(fold_result["validation_margin"])
        test_margins.append(fold_result["test_margin"])
        epoch_logs.extend(fold_result["epoch_logs"])
        fold_records.append(
            {
                "backbone": backbone_key,
                "learning_rate": learning_rate,
                "precision": precision,
                "fold": fold_index,
                "n_fit_rows": int(len(fit_index)),
                "n_holdout_rows": int(len(holdout_index)),
                "best_epoch": fold_result["best_epoch"],
                "holdout_accuracy": fold_result["best_holdout_accuracy"],
                "runtime_seconds": fold_result["runtime_seconds"],
            }
        )

    if np.isnan(oof_margin).any():
        raise RuntimeError(f"{backbone_key}: OOF margin vector has gaps; fold coverage is incomplete")

    oof_probability = margins_to_probabilities(oof_margin)
    oof_accuracy = float(
        accuracy_score(y_train, (oof_probability >= DEFAULT_DECISION_THRESHOLD).astype(int))
    )

    return {
        "backbone": backbone_key,
        "model_name": spec["model_name"],
        "learning_rate": float(learning_rate),
        "precision": precision,
        "seed": int(seed),
        "oof_margin": oof_margin,
        "oof_accuracy": oof_accuracy,
        "validation_margin": np.mean(np.vstack(validation_margins), axis=0),
        "test_margin": np.mean(np.vstack(test_margins), axis=0),
        "fold_records": fold_records,
        "epoch_logs": epoch_logs,
        "runtime_seconds": time.time() - config_start,
    }


def run_backbone_cv(backbone_key: str, seed: int = RANDOM_SEED) -> Optional[dict]:
    """
    Select the learning rate for one backbone on held-out TRAIN folds, with precision fallback.

    Learning rates are compared by OOF accuracy on the 2099 train rows, never on validation.
    If any precision produces a non-finite loss the whole backbone is retried at the next
    precision in the chain; if every precision fails, None is returned so the caller can
    record the drop and continue with the remaining backbones.
    """
    spec = BACKBONES[backbone_key]
    for precision in precision_fallback_sequence(spec["start_precision"]):
        try:
            lr_results = []
            for learning_rate in spec["learning_rates"]:
                print("\n" + "-" * 72)
                print(f"{backbone_key} ({spec['model_name']}) | lr={learning_rate:g} | precision={precision}")
                lr_results.append(run_backbone_at_learning_rate(backbone_key, learning_rate, precision, seed))
                print(
                    f"{backbone_key} | lr={learning_rate:g} | train-side OOF accuracy = "
                    f"{lr_results[-1]['oof_accuracy']:.4f}"
                )

            selected = max(lr_results, key=lambda r: r["oof_accuracy"])
            selected["learning_rate_candidates"] = [
                {"learning_rate": r["learning_rate"], "oof_accuracy": r["oof_accuracy"]} for r in lr_results
            ]
            selected["fold_records"] = [rec for r in lr_results for rec in r["fold_records"]]
            selected["epoch_logs"] = [rec for r in lr_results for rec in r["epoch_logs"]]
            print(
                f"{backbone_key}: selected lr={selected['learning_rate']:g} at {precision} "
                f"(OOF accuracy {selected['oof_accuracy']:.4f})"
            )
            return selected
        except NanLossError as error:
            print(f"[{backbone_key}] {error}")
            print(f"[{backbone_key}] falling back to the next precision in {precision_fallback_sequence(spec['start_precision'])}")
            free_gpu_memory()
            continue
    return None

In [ ]:
# ============================================================
# 11. Train the four backbones sequentially
# ============================================================

member_results: Dict[str, dict] = {}
dropped_backbones: Dict[str, str] = {}
all_fold_records: List[dict] = []
all_epoch_logs: List[dict] = []

for backbone_key in BACKBONE_ORDER:
    spec = BACKBONES[backbone_key]
    elapsed_seconds = time.time() - RUN_START_TIME

    if spec["droppable_on_time_budget"] and elapsed_seconds > MAX_ELAPSED_SECONDS_BEFORE_DROPPING_OPTIONAL_BACKBONE:
        reason = (
            f"time budget: {elapsed_seconds / SECONDS_PER_HOUR:.2f} h elapsed exceeds the "
            f"{MAX_ELAPSED_SECONDS_BEFORE_DROPPING_OPTIONAL_BACKBONE / SECONDS_PER_HOUR:.0f} h abort criterion"
        )
        dropped_backbones[backbone_key] = reason
        print(f"\nDROPPING {backbone_key} ({spec['model_name']}): {reason}")
        continue

    print("\n" + "=" * 80)
    print(f"BACKBONE {backbone_key}: {spec['model_name']}  ({spec['params_millions']}M, {spec['role']})")
    print(f"elapsed so far: {elapsed_seconds / SECONDS_PER_HOUR:.2f} h")

    result = run_backbone_cv(backbone_key)
    if result is None:
        reason = "non-finite loss under every precision in the fallback chain (fp16 -> bf16 -> fp32)"
        dropped_backbones[backbone_key] = reason
        print(f"DROPPING {backbone_key}: {reason}")
        free_gpu_memory()
        continue

    member_results[backbone_key] = result
    all_fold_records.extend(result["fold_records"])
    all_epoch_logs.extend(result["epoch_logs"])
    free_gpu_memory()

if not member_results:
    raise RuntimeError("No backbone trained successfully; there is nothing to evaluate.")

pd.DataFrame(all_fold_records).to_csv(OUTPUT_DIR / "training_logs" / "fold_records.csv", index=False)
pd.DataFrame(all_epoch_logs).to_csv(OUTPUT_DIR / "training_logs" / "epoch_logs.csv", index=False)

backbone_summary = {
    key: {
        "model_name": result["model_name"],
        "selected_learning_rate": result["learning_rate"],
        "precision": result["precision"],
        "seed": result["seed"],
        "train_oof_accuracy": result["oof_accuracy"],
        "learning_rate_candidates": result["learning_rate_candidates"],
        "runtime_seconds": result["runtime_seconds"],
    }
    for key, result in member_results.items()
}
save_json(
    {"backbones": backbone_summary, "dropped": dropped_backbones, "run_id": RUN_ID},
    OUTPUT_DIR / "backbone_training_summary.json",
)

print("\n" + "=" * 80)
print("Trained backbones:", sorted(member_results))
print("Dropped backbones:", dropped_backbones if dropped_backbones else "none")
for key, summary in backbone_summary.items():
    print(
        f"  {key}: lr={summary['selected_learning_rate']:g} precision={summary['precision']} "
        f"train_OOF_acc={summary['train_oof_accuracy']:.4f} "
        f"({summary['runtime_seconds'] / 60:.1f} min)"
    )
print(f"Total elapsed: {(time.time() - RUN_START_TIME) / SECONDS_PER_HOUR:.2f} h")

In [ ]:
# ============================================================
# 12. Apply exclusion rules and build candidates C1-C7
# ============================================================

# Abort criterion: a backbone whose train-side OOF accuracy is below the floor is
# excluded from the ensemble candidates. It is NOT retuned, and it still contributes
# its own single-model candidate row and its own probability artifact.
excluded_from_ensemble = {
    key: result["oof_accuracy"]
    for key, result in member_results.items()
    if result["oof_accuracy"] < MIN_OOF_ACCURACY_FOR_ENSEMBLE
}
ensemble_members = [
    key for key in BACKBONE_ORDER if key in member_results and key not in excluded_from_ensemble
]

print("Ensemble members (C5/C6/C7):", ensemble_members)
if excluded_from_ensemble:
    for key, accuracy in excluded_from_ensemble.items():
        print(f"  excluded {key}: train OOF accuracy {accuracy:.4f} < {MIN_OOF_ACCURACY_FOR_ENSEMBLE}")

ensemble_available = len(ensemble_members) >= MIN_MEMBERS_FOR_ENSEMBLE
if not ensemble_available:
    print(
        f"Fewer than {MIN_MEMBERS_FOR_ENSEMBLE} eligible members: C5, C6 and C7 are unavailable "
        "and only the single-backbone candidates will be scored."
    )


def stack_matrix(keys: Sequence[str], margin_field: str) -> np.ndarray:
    """Column-stack member margins in fixed BACKBONE_ORDER so stack coefficients stay interpretable."""
    return np.column_stack([member_results[key][margin_field] for key in keys])


candidates: Dict[str, dict] = {}

# C1-C4: each backbone alone, fold-bagged at the logit level.
for candidate_id, backbone_key in CANDIDATE_TO_SINGLE_BACKBONE.items():
    if backbone_key not in member_results:
        continue
    result = member_results[backbone_key]
    candidates[candidate_id] = {
        "candidate_id": candidate_id,
        "description": f"{backbone_key} alone ({result['model_name']}), mean of {N_FOLDS} fold logits",
        "members": [backbone_key],
        "validation_prob": margins_to_probabilities(result["validation_margin"]),
        "test_prob": margins_to_probabilities(result["test_margin"]),
        "train_oof_accuracy": result["oof_accuracy"],
    }

stack_model = None
if ensemble_available:
    oof_matrix = stack_matrix(ensemble_members, "oof_margin")
    validation_matrix = stack_matrix(ensemble_members, "validation_margin")
    test_matrix = stack_matrix(ensemble_members, "test_margin")

    # C5: equal-weight mean of member LOGITS (margins), then one sigmoid.
    candidates["C5"] = {
        "candidate_id": "C5",
        "description": "equal-weight mean of member logits",
        "members": list(ensemble_members),
        "validation_prob": margins_to_probabilities(validation_matrix.mean(axis=1)),
        "test_prob": margins_to_probabilities(test_matrix.mean(axis=1)),
        "train_oof_accuracy": float(
            accuracy_score(
                y_train,
                (margins_to_probabilities(oof_matrix.mean(axis=1)) >= DEFAULT_DECISION_THRESHOLD).astype(int),
            )
        ),
    }

    # C6: equal-weight mean of member PROBABILITIES.
    candidates["C6"] = {
        "candidate_id": "C6",
        "description": "equal-weight mean of member probabilities",
        "members": list(ensemble_members),
        "validation_prob": margins_to_probabilities(validation_matrix).mean(axis=1),
        "test_prob": margins_to_probabilities(test_matrix).mean(axis=1),
        "train_oof_accuracy": float(
            accuracy_score(
                y_train,
                (margins_to_probabilities(oof_matrix).mean(axis=1) >= DEFAULT_DECISION_THRESHOLD).astype(int),
            )
        ),
    }

    # C7: logistic stack over the OOF logit columns, fit on the 2099 TRAIN rows only.
    # This is where the combination rule is learned -- deliberately not on validation,
    # which is the methodological defect this cycle repairs relative to 08.
    stack_model = LogisticRegression(
        C=STACK_INVERSE_REGULARIZATION,
        max_iter=STACK_MAX_ITERATIONS,
        solver="lbfgs",
    )
    stack_model.fit(oof_matrix, y_train)
    candidates["C7"] = {
        "candidate_id": "C7",
        "description": f"logistic stack over {len(ensemble_members)} OOF logit columns, fit on {len(train_df)} train rows",
        "members": list(ensemble_members),
        "validation_prob": stack_model.predict_proba(validation_matrix)[:, POSITIVE_CLASS_INDEX],
        "test_prob": stack_model.predict_proba(test_matrix)[:, POSITIVE_CLASS_INDEX],
        "train_oof_accuracy": float(
            accuracy_score(y_train, stack_model.predict(oof_matrix))
        ),
    }
    stack_coefficients = {
        key: float(coefficient)
        for key, coefficient in zip(ensemble_members, stack_model.coef_[0])
    }
    save_json(
        {
            "members_in_column_order": ensemble_members,
            "coefficients": stack_coefficients,
            "intercept": float(stack_model.intercept_[0]),
            "n_parameters": int(stack_model.coef_.size + 1),
            "fit_rows": int(len(train_df)),
            "fit_split": "train (out-of-fold logits)",
        },
        OUTPUT_DIR / "logit_stack_coefficients.json",
    )
    print("Stack coefficients (fit on train OOF logits):", stack_coefficients)
    print("Stack intercept:", float(stack_model.intercept_[0]))

missing_candidates = [c for c in CANDIDATE_IDS if c not in candidates]
print("Available candidates:", sorted(candidates))
if missing_candidates:
    print("Unavailable candidates (backbone dropped or too few members):", missing_candidates)

In [ ]:
# ============================================================
# 13. Score the declared candidates on validation and select the winner
# ============================================================

# This cell is the entire validation budget except for the single threshold sweep in
# section 14: seven declared candidates, each scored once at the fixed default threshold.
# Ablation-only rows (section 17) are recorded but are never eligible for selection.

VALIDATION_LOG_PATH = OUTPUT_DIR / "val_log.jsonl"

candidate_rows = []
for candidate_id in CANDIDATE_IDS:
    if candidate_id not in candidates:
        continue
    candidate = candidates[candidate_id]
    metrics = compute_binary_metrics(
        y_val, candidate["validation_prob"], DEFAULT_DECISION_THRESHOLD, positive_label
    )
    row = {
        "candidate_id": candidate_id,
        "description": candidate["description"],
        "members": "+".join(candidate["members"]),
        "train_oof_accuracy": candidate["train_oof_accuracy"],
        "validation_accuracy": metrics["accuracy"],
        "validation_balanced_accuracy": metrics["balanced_accuracy"],
        "validation_positive_f1": metrics["positive_f1"],
        "validation_roc_auc": metrics["roc_auc"],
        "validation_pr_auc": metrics["pr_auc"],
        "scored_at_threshold": DEFAULT_DECISION_THRESHOLD,
    }
    candidate_rows.append(row)
    append_jsonl({"run_id": RUN_ID, "stage": "candidate_scoring", **row}, VALIDATION_LOG_PATH)

candidate_scores = pd.DataFrame(candidate_rows)
candidate_scores = candidate_scores.sort_values(
    by=["validation_accuracy", "validation_roc_auc"], ascending=[False, False]
).reset_index(drop=True)
candidate_scores.to_csv(OUTPUT_DIR / "ablation" / "candidate_scores_validation.csv", index=False)

# Selection rule: argmax validation accuracy; ties broken by higher validation ROC-AUC.
winning_candidate_id = str(candidate_scores.iloc[0]["candidate_id"])
winning_candidate = candidates[winning_candidate_id]

print(candidate_scores.to_string(index=False))
print("\nSelection rule: argmax validation accuracy, ties broken by validation ROC-AUC.")
print("Winning candidate:", winning_candidate_id, "-", winning_candidate["description"])
append_jsonl(
    {"run_id": RUN_ID, "stage": "selection", "winner": winning_candidate_id},
    VALIDATION_LOG_PATH,
)

In [ ]:
# ============================================================
# 14. Threshold grid applied once to the winner; validation metrics
# ============================================================

# The grid is swept exactly once, on the winner, for accuracy -- the metric this
# technique is judged on. Thresholding for positive_f1 while being promoted on
# accuracy is the mismatch this notebook deliberately avoids.
selected_threshold, threshold_sweep = select_threshold(
    y_val,
    winning_candidate["validation_prob"],
    THRESHOLD_GRID,
    metric=THRESHOLD_OBJECTIVE_METRIC,
)
threshold_sweep.sort_values("threshold").to_csv(
    OUTPUT_DIR / "threshold_sweep_validation.csv", index=False
)

validation_metrics = compute_binary_metrics(
    y_val, winning_candidate["validation_prob"], selected_threshold, positive_label
)
validation_metrics_record = {
    "technique": TECHNIQUE_ID,
    "split": "validation",
    "run_id": RUN_ID,
    "candidate_id": winning_candidate_id,
    "candidate_description": winning_candidate["description"],
    "members": winning_candidate["members"],
    "split_version": SPLIT_VERSION,
    "dataset_version": DATASET_VERSION,
    "threshold_strategy": f"grid {THRESHOLD_GRID[0]}-{THRESHOLD_GRID[-1]} maximizing validation {THRESHOLD_OBJECTIVE_METRIC}",
    **validation_metrics,
}
save_json(validation_metrics_record, OUTPUT_DIR / "metrics_validation.json")

# Validation-only selection record. best_config.json is written later, and only if
# the gate passes, because a locked configuration without a test unlock is not a "best config".
save_json(
    {
        "technique": TECHNIQUE_ID,
        "run_id": RUN_ID,
        "selected_candidate": winning_candidate_id,
        "selected_threshold": selected_threshold,
        "selection_metric": "validation accuracy (ties: validation roc_auc)",
        "threshold_objective": THRESHOLD_OBJECTIVE_METRIC,
        "test_set_used_for_selection": False,
        "n_validation_candidates_declared": len(CANDIDATE_IDS),
        "n_validation_candidates_scored": int(len(candidate_scores)),
        "backbones": backbone_summary,
        "dropped_backbones": dropped_backbones,
        "excluded_from_ensemble": excluded_from_ensemble,
    },
    OUTPUT_DIR / "selected_candidate_validation.json",
)

validation_predictions = make_sanitized_prediction_frame(
    val_df, winning_candidate["validation_prob"], selected_threshold
)
validation_predictions.to_csv(
    OUTPUT_DIR / "error_analysis" / "error_analysis_validation.csv", index=False
)

print("Selected threshold:", selected_threshold)
print("Validation accuracy:", round(validation_metrics["accuracy"], 4))
print("Validation balanced accuracy:", round(validation_metrics["balanced_accuracy"], 4))
print("Validation ROC-AUC:", round(validation_metrics["roc_auc"], 4))
print("Validation positive F1:", round(validation_metrics["positive_f1"], 4))
print("Comparator 07 validation accuracy:", COMPARATOR_VALIDATION_ACCURACY,
      "| ROC-AUC:", COMPARATOR_VALIDATION_ROC_AUC)

In [ ]:
# ============================================================
# 15. Validation gate decision
# ============================================================

# Preregistered rule: unlock test only if the winner clears BOTH thresholds.
# The accuracy gate is the minimum validation result consistent with the 90% test goal
# under the repository's measured val->test offset (+2.83pp, sd 0.38, n=7). The AUC gate
# forces the gain to show up in ranking quality rather than at one lucky threshold.

gate_accuracy = float(validation_metrics["accuracy"])
gate_roc_auc = float(validation_metrics["roc_auc"])
gate_accuracy_met = gate_accuracy >= VALIDATION_GATE_MIN_ACCURACY
gate_roc_auc_met = gate_roc_auc >= VALIDATION_GATE_MIN_ROC_AUC
TEST_UNLOCKED = bool(gate_accuracy_met and gate_roc_auc_met)

gate_record = {
    "technique": TECHNIQUE_ID,
    "run_id": RUN_ID,
    "candidate_id": winning_candidate_id,
    "selected_threshold": selected_threshold,
    "validation_accuracy": gate_accuracy,
    "validation_accuracy_required": VALIDATION_GATE_MIN_ACCURACY,
    "validation_accuracy_met": gate_accuracy_met,
    "validation_roc_auc": gate_roc_auc,
    "validation_roc_auc_required": VALIDATION_GATE_MIN_ROC_AUC,
    "validation_roc_auc_met": gate_roc_auc_met,
    "test_unlocked": TEST_UNLOCKED,
}
save_json(gate_record, OUTPUT_DIR / "validation_gate.json")
append_jsonl({"run_id": RUN_ID, "stage": "validation_gate", **gate_record}, VALIDATION_LOG_PATH)

BANNER = "#" * 78
print(BANNER)
if TEST_UNLOCKED:
    print("## VALIDATION GATE MET")
    print(f"## validation accuracy {gate_accuracy:.4f} >= {VALIDATION_GATE_MIN_ACCURACY}")
    print(f"## validation ROC-AUC  {gate_roc_auc:.4f} >= {VALIDATION_GATE_MIN_ROC_AUC}")
    print("## The held-out test split will be evaluated ONCE, for ONE configuration,")
    print("## at ONE threshold, in section 18. This consumes this technique's only")
    print("## test-budget slot and raises the Holm correction for every future candidate.")
else:
    print("## GATE NOT MET  --  TEST SPLIT REMAINS LOCKED")
    print(f"## validation accuracy {gate_accuracy:.4f} vs required {VALIDATION_GATE_MIN_ACCURACY}"
          f"  [{'ok' if gate_accuracy_met else 'FAIL'}]")
    print(f"## validation ROC-AUC  {gate_roc_auc:.4f} vs required {VALIDATION_GATE_MIN_ROC_AUC}"
          f"  [{'ok' if gate_roc_auc_met else 'FAIL'}]")
    print("## No test metric will be computed, printed or saved by this notebook.")
    print("## Per the preregistered stop rule, this closes the architecture line of work:")
    print("## write the validation-only result set, commit the probability artifacts,")
    print("## and redirect to the label-quality audit of the rows every model misses.")
print(BANNER)

In [ ]:
# ============================================================
# 16. Sanitized probability export  (runs whether or not the gate passed)
# ============================================================

# These CSVs are the highest-leverage artifact this run produces: they make calibration,
# thresholding, cross-technique ensembling and every paired statistical test runnable on
# CPU with no GPU and no human gate. They carry exactly four columns and no raw text.

PROBS_SCHEMA_COLUMNS = ["row_id", "split", "y_true", "y_prob"]
TEXT_BEARING_COLUMNS = {"text", "text_hash", "raw_text", "text_preview", "content"}
PRIMARY_ARTIFACT_NAME = TECHNIQUE_ID
MEMBER_ARTIFACT_PREFIX = "09_MEMBER-"

EXPORT_SPLITS = {"validation": val_df, "test": test_df}


def write_probability_artifact(artifact_name: str, split_name: str, y_prob) -> Path:
    """
    Write <artifact_name>__<split>.csv with exactly row_id, split, y_true, y_prob.

    Asserts before writing that the row_id set equals the split's row_id set, that the row
    count matches the frozen split, and that no text-bearing column can reach the file.
    Loading tools reject any probability file carrying text, so a leak fails loudly at load;
    these asserts make it fail even earlier, at write time.
    """
    split_frame = EXPORT_SPLITS[split_name]
    frame = pd.DataFrame(
        {
            "row_id": split_frame[id_col].values,
            "split": split_name,
            "y_true": split_frame[label_col].values.astype(int),
            "y_prob": np.asarray(y_prob, dtype=float),
        }
    )

    if list(frame.columns) != PROBS_SCHEMA_COLUMNS:
        raise RuntimeError(f"{artifact_name}/{split_name}: column set {list(frame.columns)} != {PROBS_SCHEMA_COLUMNS}")
    leaking = TEXT_BEARING_COLUMNS & set(frame.columns)
    if leaking:
        raise RuntimeError(f"{artifact_name}/{split_name}: refusing to export text-bearing columns {leaking}")
    if set(frame["row_id"]) != set(split_frame[id_col].tolist()):
        raise RuntimeError(f"{artifact_name}/{split_name}: row_id set does not match the {split_name} split")
    if len(frame) != EXPECTED_SPLIT_ROW_COUNTS[split_name]:
        raise RuntimeError(
            f"{artifact_name}/{split_name}: {len(frame)} rows, expected {EXPECTED_SPLIT_ROW_COUNTS[split_name]}"
        )
    if frame["y_prob"].isna().any() or not np.isfinite(frame["y_prob"].values).all():
        raise RuntimeError(f"{artifact_name}/{split_name}: non-finite probabilities")
    if frame["y_prob"].min() < 0.0 or frame["y_prob"].max() > 1.0:
        raise RuntimeError(f"{artifact_name}/{split_name}: probabilities outside [0, 1]")

    path = PROBS_DIR / f"{artifact_name}__{split_name}.csv"
    frame.to_csv(path, index=False)
    return path


def write_probability_meta(artifact_name: str, meta_extra: dict) -> Path:
    """Write the <artifact_name>__meta.json sidecar that pairs with the probability CSVs."""
    meta = {
        "technique": TECHNIQUE_ID,
        "artifact": artifact_name,
        "run_id": RUN_ID,
        "created_at_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
        "random_seed": RANDOM_SEED,
        "fold_seed": RANDOM_SEED,
        "n_folds": N_FOLDS,
        "split_version": SPLIT_VERSION,
        "dataset_version": DATASET_VERSION,
        "schema": PROBS_SCHEMA_COLUMNS,
        "contains_raw_text": False,
        "row_counts": {split_name: EXPECTED_SPLIT_ROW_COUNTS[split_name] for split_name in EXPORT_SPLITS},
        **meta_extra,
    }
    path = PROBS_DIR / f"{artifact_name}__meta.json"
    save_json(meta, path)
    return path


exported_paths = []

# Primary artifact: the winning candidate.
for split_name in EXPORT_SPLITS:
    prob_key = "validation_prob" if split_name == "validation" else "test_prob"
    exported_paths.append(write_probability_artifact(PRIMARY_ARTIFACT_NAME, split_name, winning_candidate[prob_key]))
exported_paths.append(
    write_probability_meta(
        PRIMARY_ARTIFACT_NAME,
        {
            "candidate_id": winning_candidate_id,
            "candidate_description": winning_candidate["description"],
            "members": winning_candidate["members"],
            "checkpoints": {key: BACKBONES[key]["model_name"] for key in winning_candidate["members"]},
            "member_seeds": {key: member_results[key]["seed"] for key in winning_candidate["members"]},
            "member_learning_rates": {key: member_results[key]["learning_rate"] for key in winning_candidate["members"]},
            "member_precisions": {key: member_results[key]["precision"] for key in winning_candidate["members"]},
            "selected_threshold": selected_threshold,
            "threshold_objective": THRESHOLD_OBJECTIVE_METRIC,
            "validation_gate_passed": TEST_UNLOCKED,
        },
    )
)

# Per-member artifacts: one per trained backbone, whether or not it entered the ensemble.
for backbone_key, result in member_results.items():
    artifact_name = f"{MEMBER_ARTIFACT_PREFIX}{backbone_key}"
    exported_paths.append(
        write_probability_artifact(artifact_name, "validation", margins_to_probabilities(result["validation_margin"]))
    )
    exported_paths.append(
        write_probability_artifact(artifact_name, "test", margins_to_probabilities(result["test_margin"]))
    )
    exported_paths.append(
        write_probability_meta(
            artifact_name,
            {
                "member": backbone_key,
                "checkpoint": result["model_name"],
                "seed": result["seed"],
                "learning_rate": result["learning_rate"],
                "precision": result["precision"],
                "train_oof_accuracy": result["oof_accuracy"],
                "in_ensemble": backbone_key in ensemble_members,
                "train_oof_rows": EXPECTED_SPLIT_ROW_COUNTS["train"],
                "aggregation": f"mean of {N_FOLDS} fold logits, sigmoid of the mean margin",
            },
        )
    )

# The out-of-fold train probabilities let a future CPU-only cycle refit any stacker
# without a GPU. Same sanitized schema.
for backbone_key, result in member_results.items():
    artifact_name = f"{MEMBER_ARTIFACT_PREFIX}{backbone_key}"
    oof_frame = pd.DataFrame(
        {
            "row_id": train_df[id_col].values,
            "split": "train",
            "y_true": y_train,
            "y_prob": margins_to_probabilities(result["oof_margin"]),
        }
    )
    if list(oof_frame.columns) != PROBS_SCHEMA_COLUMNS:
        raise RuntimeError(f"{artifact_name}/train: unexpected columns {list(oof_frame.columns)}")
    if len(oof_frame) != EXPECTED_SPLIT_ROW_COUNTS["train"]:
        raise RuntimeError(f"{artifact_name}/train: {len(oof_frame)} rows, expected {EXPECTED_SPLIT_ROW_COUNTS['train']}")
    oof_path = PROBS_DIR / f"{artifact_name}__train_oof.csv"
    oof_frame.to_csv(oof_path, index=False)
    exported_paths.append(oof_path)

print(f"Exported {len(exported_paths)} sanitized probability artifacts to {PROBS_DIR}:")
for path in exported_paths:
    print(" -", path.name)
print("\nAfter the run, download these and place them in research_loop/probs/ in the repository.")

In [ ]:
# ============================================================
# 17. Validation-only ablation table
# ============================================================

# Every row here is validation-only and preregistered in the ablation_plan. None of it
# is eligible for selection: the winner was already fixed in section 13.

ABLATION_MODEL_FAMILY = CONFIG["model_family"]
ABLATION_FEATURE_FAMILY = CONFIG["feature_family"]


def ablation_row(config_id: str, notes: str, y_prob=None, hyperparameters: str = "",
                 train_oof_accuracy=np.nan, delta_validation_accuracy=np.nan, selectable: bool = False) -> dict:
    """Build one ablation_results.csv row, scoring y_prob on validation at the default threshold when given."""
    metrics = (
        compute_binary_metrics(y_val, y_prob, DEFAULT_DECISION_THRESHOLD, positive_label)
        if y_prob is not None
        else {}
    )
    return {
        "technique": TECHNIQUE_ID,
        "config_id": config_id,
        "model_family": ABLATION_MODEL_FAMILY,
        "feature_family": ABLATION_FEATURE_FAMILY,
        "hyperparameter_summary": hyperparameters,
        "threshold": DEFAULT_DECISION_THRESHOLD if y_prob is not None else np.nan,
        "validation_accuracy": metrics.get("accuracy", np.nan),
        "validation_positive_f1": metrics.get("positive_f1", np.nan),
        "validation_roc_auc": metrics.get("roc_auc", np.nan),
        "validation_pr_auc": metrics.get("pr_auc", np.nan),
        "train_oof_accuracy": train_oof_accuracy,
        "delta_validation_accuracy": delta_validation_accuracy,
        "selectable_candidate": selectable,
        "notes": notes,
    }


ablation_rows = []

# --- the seven declared candidates, plus per-backbone OOF accuracy alongside ----------
for candidate_id in CANDIDATE_IDS:
    if candidate_id not in candidates:
        ablation_rows.append(
            ablation_row(
                candidate_id,
                notes="not available: backbone dropped or too few eligible ensemble members",
            )
        )
        continue
    candidate = candidates[candidate_id]
    backbone_key = CANDIDATE_TO_SINGLE_BACKBONE.get(candidate_id)
    hyperparameters = ""
    if backbone_key is not None:
        result = member_results[backbone_key]
        hyperparameters = (
            f"{result['model_name']}, lr={result['learning_rate']:g}, precision={result['precision']}, "
            f"batch={TRAIN_BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS}, max_len={MAX_LENGTH}, folds={N_FOLDS}"
        )
    else:
        hyperparameters = f"members={'+'.join(candidate['members'])}, folds={N_FOLDS}"
    ablation_rows.append(
        ablation_row(
            candidate_id,
            notes=candidate["description"],
            y_prob=candidate["validation_prob"],
            hyperparameters=hyperparameters,
            train_oof_accuracy=candidate["train_oof_accuracy"],
            selectable=True,
        )
    )


def candidate_validation_accuracy(candidate_id: str) -> float:
    """Validation accuracy of a candidate at the default threshold, or NaN when unavailable."""
    if candidate_id not in candidates:
        return float("nan")
    return float(
        compute_binary_metrics(
            y_val, candidates[candidate_id]["validation_prob"], DEFAULT_DECISION_THRESHOLD, positive_label
        )["accuracy"]
    )


# --- the four preregistered contrasts --------------------------------------------------
CONTRASTS = [
    ("C1", "C3", "SCALE at fixed family and corpus: 355M twitter-roberta-large-hate vs the 125M champion. "
                 "This is the central hypothesis and the scientific payload even if every ensemble fails."),
    ("C3", "C4", "PRETRAINING DOMAIN at fixed capacity: two 125M hate-speech checkpoints, different corpora."),
    ("C5", "C6", "logit-level vs probability-level averaging, previously untried in this repository."),
    ("C5", "C7", "equal weights vs weights learned on 2099 train OOF rows."),
]
for left, right, rationale in CONTRASTS:
    left_accuracy = candidate_validation_accuracy(left)
    right_accuracy = candidate_validation_accuracy(right)
    ablation_rows.append(
        ablation_row(
            f"CONTRAST_{left}_vs_{right}",
            notes=(
                f"{rationale} {left} val acc={left_accuracy:.4f}, {right} val acc={right_accuracy:.4f}, "
                f"delta={100 * (left_accuracy - right_accuracy):+.2f}pp"
            ),
            delta_validation_accuracy=left_accuracy - right_accuracy,
        )
    )

# --- C7 plus a fifth word-TF-IDF OOF column -------------------------------------------
# Preregistered prediction: +0.2pp or less. Recorded to falsify the lexical-complementarity
# hypothesis on the record rather than by assumption. Ablation only, never selectable.
tfidf_ablation_note = "not run: C7 unavailable"
if "C7" in candidates:
    tfidf_folder = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    tfidf_oof_margin = np.full(len(train_df), np.nan, dtype=float)
    train_texts = train_df[text_col].values

    for fit_index, holdout_index in tfidf_folder.split(np.zeros(len(train_df)), y_train):
        vectorizer = TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE, min_df=TFIDF_MIN_DF, sublinear_tf=True
        )
        fold_features = vectorizer.fit_transform(train_texts[fit_index])
        fold_model = LogisticRegression(
            C=TFIDF_INVERSE_REGULARIZATION, max_iter=STACK_MAX_ITERATIONS, class_weight="balanced"
        )
        fold_model.fit(fold_features, y_train[fit_index])
        tfidf_oof_margin[holdout_index] = fold_model.decision_function(vectorizer.transform(train_texts[holdout_index]))

    full_vectorizer = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE, min_df=TFIDF_MIN_DF, sublinear_tf=True
    )
    full_features = full_vectorizer.fit_transform(train_texts)
    full_model = LogisticRegression(
        C=TFIDF_INVERSE_REGULARIZATION, max_iter=STACK_MAX_ITERATIONS, class_weight="balanced"
    )
    full_model.fit(full_features, y_train)
    tfidf_validation_margin = full_model.decision_function(full_vectorizer.transform(val_df[text_col].values))

    extended_oof = np.column_stack([stack_matrix(ensemble_members, "oof_margin"), tfidf_oof_margin])
    extended_validation = np.column_stack(
        [stack_matrix(ensemble_members, "validation_margin"), tfidf_validation_margin]
    )
    extended_stack = LogisticRegression(
        C=STACK_INVERSE_REGULARIZATION, max_iter=STACK_MAX_ITERATIONS, solver="lbfgs"
    )
    extended_stack.fit(extended_oof, y_train)
    extended_validation_prob = extended_stack.predict_proba(extended_validation)[:, POSITIVE_CLASS_INDEX]

    c7_accuracy = candidate_validation_accuracy("C7")
    extended_accuracy = float(
        compute_binary_metrics(y_val, extended_validation_prob, DEFAULT_DECISION_THRESHOLD, positive_label)["accuracy"]
    )
    tfidf_ablation_note = (
        f"C7 plus a fifth word-TF-IDF OOF logit column. Preregistered prediction: +0.2pp or less. "
        f"Observed {100 * (extended_accuracy - c7_accuracy):+.2f}pp vs C7 ({c7_accuracy:.4f})."
    )
    ablation_rows.append(
        ablation_row(
            "ABL_C7_PLUS_WORD_TFIDF",
            notes=tfidf_ablation_note,
            y_prob=extended_validation_prob,
            hyperparameters=(
                f"stack over {len(ensemble_members)} transformer OOF logits + 1 word-TF-IDF OOF logit "
                f"(ngram={TFIDF_NGRAM_RANGE}, min_df={TFIDF_MIN_DF}, max_features={TFIDF_MAX_FEATURES})"
            ),
            train_oof_accuracy=float(accuracy_score(y_train, extended_stack.predict(extended_oof))),
            delta_validation_accuracy=extended_accuracy - c7_accuracy,
        )
    )
print(tfidf_ablation_note)

# --- drops and exclusions, on the record ----------------------------------------------
for backbone_key, reason in dropped_backbones.items():
    ablation_rows.append(
        ablation_row(
            f"DROPPED_{backbone_key}",
            notes=f"{BACKBONES[backbone_key]['model_name']} dropped: {reason}",
        )
    )
for backbone_key, oof_accuracy in excluded_from_ensemble.items():
    ablation_rows.append(
        ablation_row(
            f"EXCLUDED_{backbone_key}",
            notes=(
                f"{BACKBONES[backbone_key]['model_name']} excluded from C5/C6/C7: train OOF accuracy "
                f"{oof_accuracy:.4f} < {MIN_OOF_ACCURACY_FOR_ENSEMBLE}. Not retuned, per the abort criteria."
            ),
            train_oof_accuracy=oof_accuracy,
        )
    )

# --- optional 3-seed replicate of the best single backbone -----------------------------
# Prereg: "optional if time permits". Reproduces 08's seed-variation design at large scale
# as a control. Off by default; each extra seed costs one more full CV run.
if RUN_OPTIONAL_SEED_REPLICATE:
    single_candidate_scores = {
        candidate_id: candidate_validation_accuracy(candidate_id)
        for candidate_id in CANDIDATE_TO_SINGLE_BACKBONE
        if candidate_id in candidates
    }
    best_single_candidate = max(single_candidate_scores, key=single_candidate_scores.get)
    best_single_backbone = CANDIDATE_TO_SINGLE_BACKBONE[best_single_candidate]
    for replicate_seed in OPTIONAL_REPLICATE_SEEDS:
        elapsed_seconds = time.time() - RUN_START_TIME
        if elapsed_seconds > MAX_ELAPSED_SECONDS_BEFORE_DROPPING_OPTIONAL_BACKBONE:
            print(f"Skipping seed replicate {replicate_seed}: {elapsed_seconds / SECONDS_PER_HOUR:.2f} h elapsed")
            break
        replicate = run_backbone_cv(best_single_backbone, seed=replicate_seed)
        free_gpu_memory()
        if replicate is None:
            continue
        ablation_rows.append(
            ablation_row(
                f"REPLICATE_{best_single_backbone}_seed{replicate_seed}",
                notes=(
                    f"seed replicate of the best single backbone {best_single_backbone}; "
                    f"seed {RANDOM_SEED} val acc={single_candidate_scores[best_single_candidate]:.4f}"
                ),
                y_prob=margins_to_probabilities(replicate["validation_margin"]),
                hyperparameters=f"{replicate['model_name']}, lr={replicate['learning_rate']:g}, seed={replicate_seed}",
                train_oof_accuracy=replicate["oof_accuracy"],
            )
        )

ablation_results = pd.DataFrame(ablation_rows)
ablation_results.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)

print("\nAblation rows written:", len(ablation_results))
print(
    ablation_results[
        ["config_id", "validation_accuracy", "validation_positive_f1", "validation_roc_auc", "train_oof_accuracy"]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# 18. Held-out test evaluation  --  guarded by the validation gate
# ============================================================

# Runs only when section 15 unlocked test. One configuration, one threshold, once.
# If the gate failed, nothing about the test split is computed, printed or saved here.

if not TEST_UNLOCKED:
    print("GATE NOT MET: the test split stays locked.")
    print("No test metric was computed, printed or saved. The validation-only result set")
    print("and the sanitized probability artifacts are complete and are the output of this run.")
else:
    test_probabilities = winning_candidate["test_prob"]
    test_metrics = compute_binary_metrics(y_test, test_probabilities, selected_threshold, positive_label)
    test_metrics_record = {
        "technique": TECHNIQUE_ID,
        "split": "test",
        "run_id": RUN_ID,
        "candidate_id": winning_candidate_id,
        "candidate_description": winning_candidate["description"],
        "members": winning_candidate["members"],
        "split_version": SPLIT_VERSION,
        "dataset_version": DATASET_VERSION,
        "threshold_selected_on": "validation",
        "test_set_used_for_selection": False,
        "external_pretraining_used": True,
        **test_metrics,
    }
    save_json(test_metrics_record, OUTPUT_DIR / "metrics_test.json")

    test_predictions_binary = (np.asarray(test_probabilities) >= selected_threshold).astype(int)
    classification_report_test = classification_report(
        y_test,
        test_predictions_binary,
        labels=[0, 1],
        target_names=["NON_EXTREMIST", "EXTREMIST"],
        output_dict=True,
        zero_division=0,
    )
    save_json(classification_report_test, OUTPUT_DIR / "classification_report_test.json")

    confusion = confusion_matrix(y_test, test_predictions_binary, labels=[0, 1])
    confusion_long = pd.DataFrame(
        [
            {"actual": "NON_EXTREMIST", "predicted": "NON_EXTREMIST", "count": int(confusion[0, 0])},
            {"actual": "NON_EXTREMIST", "predicted": "EXTREMIST", "count": int(confusion[0, 1])},
            {"actual": "EXTREMIST", "predicted": "NON_EXTREMIST", "count": int(confusion[1, 0])},
            {"actual": "EXTREMIST", "predicted": "EXTREMIST", "count": int(confusion[1, 1])},
        ]
    )
    confusion_long.to_csv(OUTPUT_DIR / "confusion_matrix_test.csv", index=False)

    figure, axis = plt.subplots(figsize=(5, 4))
    axis.imshow(confusion)
    axis.set_xticks([0, 1])
    axis.set_yticks([0, 1])
    axis.set_xticklabels(["pred NON_EXTREMIST", "pred EXTREMIST"])
    axis.set_yticklabels(["true NON_EXTREMIST", "true EXTREMIST"])
    axis.set_title(f"{TECHNIQUE_ID} — test @ t={selected_threshold:.3f}")
    for i in range(confusion.shape[0]):
        for j in range(confusion.shape[1]):
            axis.text(j, i, str(confusion[i, j]), ha="center", va="center")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "plots" / "confusion_matrix_test.png", dpi=200, bbox_inches="tight")
    plt.close(figure)

    best_config = {
        "technique": TECHNIQUE_ID,
        "run_id": RUN_ID,
        "model_family": CONFIG["model_family"],
        "feature_family": CONFIG["feature_family"],
        "random_seed": RANDOM_SEED,
        "split_version": SPLIT_VERSION,
        "dataset_version": DATASET_VERSION,
        "hyperparameters": {
            "selected_candidate": winning_candidate_id,
            "members": winning_candidate["members"],
            "checkpoints": {key: BACKBONES[key]["model_name"] for key in winning_candidate["members"]},
            "member_learning_rates": {key: member_results[key]["learning_rate"] for key in winning_candidate["members"]},
            "member_precisions": {key: member_results[key]["precision"] for key in winning_candidate["members"]},
            "n_folds": N_FOLDS,
            "max_epochs": MAX_EPOCHS,
            "epoch_patience": EPOCH_PATIENCE,
            "batch_size": TRAIN_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "max_length": MAX_LENGTH,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "loss": "weighted_cross_entropy(class_weight=balanced)",
            "label_smoothing": LABEL_SMOOTHING,
        },
        "threshold_strategy": f"validation grid {THRESHOLD_GRID[0]}-{THRESHOLD_GRID[-1]} maximizing {THRESHOLD_OBJECTIVE_METRIC}",
        "selected_threshold": selected_threshold,
        "threshold_selected_on_validation": True,
        "test_excluded_from_model_selection": True,
        "external_pretraining_used": True,
    }
    save_json(best_config, OUTPUT_DIR / "best_config.json")

    # Secondary gates from the preregistration. Descriptive flags only: the verdict
    # string is issued by tools/compare_techniques.py, never here.
    balanced_accuracy_floor = COMPARATOR_TEST_BALANCED_ACCURACY - BALANCED_ACCURACY_NON_INFERIORITY_MARGIN
    secondary_gate_checks = {
        "technique": TECHNIQUE_ID,
        "comparator": COMPARATOR_TECHNIQUE,
        "test_accuracy": test_metrics["accuracy"],
        "comparator_test_accuracy": COMPARATOR_TEST_ACCURACY,
        "project_target_test_accuracy": PROJECT_TARGET_TEST_ACCURACY,
        "meets_project_target": bool(test_metrics["accuracy"] >= PROJECT_TARGET_TEST_ACCURACY),
        "holm_detectable_test_accuracy": HOLM_DETECTABLE_TEST_ACCURACY,
        "reaches_holm_detectable_threshold": bool(test_metrics["accuracy"] >= HOLM_DETECTABLE_TEST_ACCURACY),
        "balanced_accuracy_floor": balanced_accuracy_floor,
        "balanced_accuracy_non_inferior": bool(test_metrics["balanced_accuracy"] >= balanced_accuracy_floor),
        "false_positive_rate": test_metrics["false_positive_rate"],
        "fpr_flag_threshold": FPR_FLAG_THRESHOLD,
        "false_positive_rate_flagged": bool(test_metrics["false_positive_rate"] > FPR_FLAG_THRESHOLD),
        "verdict_authority": "tools/compare_techniques.py",
        "verdict_issued_here": None,
    }
    save_json(secondary_gate_checks, OUTPUT_DIR / "secondary_gate_checks.json")

    test_predictions = make_sanitized_prediction_frame(test_df, test_probabilities, selected_threshold)
    test_predictions.to_csv(OUTPUT_DIR / "error_analysis" / "error_analysis_test.csv", index=False)

    print("Held-out test evaluated once, for candidate", winning_candidate_id, "at threshold", selected_threshold)
    print(json.dumps(make_json_safe(test_metrics), indent=2))
    print("\nSecondary gates:", json.dumps(make_json_safe(secondary_gate_checks), indent=2))
    print("\nNo verdict is issued in this notebook. Run tools/compare_techniques.py.")

In [ ]:
# ============================================================
# 19. Experiment metadata and run card
# ============================================================

try:
    import transformers
    transformers_version = transformers.__version__
except Exception:
    transformers_version = None

try:
    import sklearn
    sklearn_version = sklearn.__version__
except Exception:
    sklearn_version = None

total_runtime_seconds = time.time() - RUN_START_TIME

metadata = {
    "run_id": RUN_ID,
    "technique": TECHNIQUE_ID,
    "technique_name": TECHNIQUE_NAME,
    "comparator": COMPARATOR_TECHNIQUE,
    "preregistration": PREREGISTRATION_PATH,
    "created_at_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
    "project_name": CONFIG["project_name"],
    "dataset_version": DATASET_VERSION,
    "split_version": SPLIT_VERSION,
    "random_seed": RANDOM_SEED,
    "processed_dataset_path": str(processed_dataset_path),
    "processed_dataset_sha256_16": sha256_file(processed_dataset_path),
    "split_assignments_path": str(split_assignments_path),
    "split_assignments_sha256_16": sha256_file(split_assignments_path),
    "dataset_manifest": dataset_manifest,
    "software_environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "sklearn_version": sklearn_version,
        "torch_version": torch.__version__,
        "transformers_version": transformers_version,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "protocol": {
        "cross_validation_scope": "train split only (2099 rows)",
        "n_folds": N_FOLDS,
        "epoch_and_lr_selection_split": "held-out train folds",
        "n_validation_candidates_declared": len(CANDIDATE_IDS),
        "n_validation_candidates_scored": int(len(candidate_scores)),
        "threshold_grids_applied_to_validation": 1,
        "threshold_objective": THRESHOLD_OBJECTIVE_METRIC,
        "test_set_used_for_selection": False,
        "test_unlocked": TEST_UNLOCKED,
    },
    "selection": {
        "winning_candidate": winning_candidate_id,
        "selected_threshold": selected_threshold,
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_roc_auc": validation_metrics["roc_auc"],
    },
    "backbones": backbone_summary,
    "dropped_backbones": dropped_backbones,
    "excluded_from_ensemble": excluded_from_ensemble,
    "runtime": {
        "total_seconds": total_runtime_seconds,
        "total_hours": total_runtime_seconds / SECONDS_PER_HOUR,
    },
    "artifacts": {
        "probability_exports": sorted(p.name for p in PROBS_DIR.glob("*")),
        "destination_in_repo": "research_loop/probs/",
    },
}
save_json(metadata, OUTPUT_DIR / "experiment_metadata.json")

readme_lines = [
    f"# {TECHNIQUE_ID}",
    "",
    f"Run ID: {RUN_ID}",
    f"Preregistration: {PREREGISTRATION_PATH}",
    f"Split: {SPLIT_VERSION}  |  Dataset: {DATASET_VERSION}  |  Seed: {RANDOM_SEED}",
    f"Winning candidate: {winning_candidate_id} ({winning_candidate['description']})",
    f"Selected threshold (validation {THRESHOLD_OBJECTIVE_METRIC}): {selected_threshold}",
    f"Validation accuracy: {validation_metrics['accuracy']:.4f}  ROC-AUC: {validation_metrics['roc_auc']:.4f}",
    f"Validation gate: {'MET -> test evaluated once' if TEST_UNLOCKED else 'NOT MET -> test left locked'}",
    f"Total runtime: {total_runtime_seconds / SECONDS_PER_HOUR:.2f} h",
    "",
    "## Where the files go",
    "- probs/*.csv and probs/*__meta.json -> research_loop/probs/",
    "- metrics_validation.json, ablation_results.csv, threshold_sweep_validation.csv,",
    "  and (only if the gate passed) metrics_test.json, classification_report_test.json,",
    "  confusion_matrix_test.csv, best_config.json -> results_summary/" + TECHNIQUE_ID + "/",
    "",
    "## Rules this run obeyed",
    "- Cross-validation folds came from the train split only; validation and test were never in a fold.",
    "- Learning rate and epoch count were selected on held-out train folds.",
    "- Validation was consumed by exactly the declared candidates plus one threshold grid.",
    "- No raw text, text hash or dataset row content is present in any probability artifact.",
    "- No verdict is issued here. tools/compare_techniques.py is the only verdict authority.",
]
(OUTPUT_DIR / "README.md").write_text("\n".join(readme_lines))

print("\n".join(readme_lines))

In [ ]:
# ============================================================
# 20. Zip artifacts for download
# ============================================================

zip_base = Path(CONFIG["output_root"]) / f"{TECHNIQUE_ID}_artifacts"
zip_path = shutil.make_archive(base_name=str(zip_base), format="zip", root_dir=OUTPUT_DIR)

print("Created artifact zip:", zip_path)
print("Contents include probs/ (the sanitized probability export), the validation result set,")
print("ablation_results.csv, training logs, and the test result set only if the gate passed.")
print("\nOpen the Kaggle Output panel after the run, download the zip, then place")
print("probs/*.csv and probs/*__meta.json into research_loop/probs/ in the repository.")

## Interpretation — read this before reading any number above

**The preregistered expectation for this run is `INCONCLUSIVE`.** The expected effect is
about +1.2pp, roughly 5 test rows out of 450. Unadjusted two-sided exact McNemar against
`07_TWITTER-ROBERTA_FINE-TUNE` needs |b−c| ≥ 14 (≈ +3.11pp, test accuracy ≥ 0.9200) to
reach significance; under Holm with a family size of 9 it needs |b−c| ≥ 20 (≈ +4.44pp,
test accuracy ≥ 0.9333). A CPU pilot found 42/450 validation rows (9.3%) misclassified by
every member of a four-model heterogeneous set, so 0.9333 sits at or above the plausible
annotation-ambiguity floor of this dataset. `INCONCLUSIVE` is a first-class, publishable
outcome here.

**If test accuracy lands in [0.9000, 0.9333) it must be reported, verbatim, as:**

> reaches the project's 90% target; NOT statistically distinguishable from
> 07_TWITTER-ROBERTA_FINE-TUNE at n=450

That sentence must appear in `results_summary/09_MULTI-CHECKPOINT_LOGIT-STACK/` and in any
documentation table footnote that cites this result. The champion is **unchanged** in that
case. Only test accuracy ≥ 0.9333 *with* McNemar exact p < 0.0056, test balanced accuracy
≥ 0.8803 and test FPR ≤ 0.12 may change the champion.

**Verdict authority.** This notebook computes metrics; it does not adjudicate. The verdict
string comes from `tools/compare_techniques.py` and from nothing else — not from this
notebook, not from an agent, not from a reader's impression of the numbers. Section 18
deliberately writes `"verdict_issued_here": null`.

**Two further cautions.**

1. The measured val→test offset across the seven existing techniques is +2.83pp (sd 0.38).
   The test split is systematically *easier* than validation for every technique measured.
   A test number above its validation counterpart is the expected behaviour of this split,
   not evidence that validation underestimated the model.
2. Per the preregistered stop rule, either outcome short of `SUPERIOR` ends the architecture
   line: 07, 08 and 09 will together have established that the plateau is a property of the
   data, not of the model family. The follow-ups are a label-quality audit of the ~9% of rows
   every model misses, the CPU-local probability-artifact infrastructure this run produces,
   and reporting the plateau itself as the finding.